# 5.3 중요도 샘플링과 MC의 한계 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter05_3_importance_sampling.ipynb)

책 본문: [5.3절](https://smhanlab.com/book-ml/kor/ml2/chapter05/3.html)

5.2절의 5칸 GridWorld에서 **행동 정책 \(b\)**(50-50 무작위)만 행동해서
데이터를 모으고, **목표 정책 \(\pi\)**(항상 오른쪽)의 가치를
추정합니다. "리턴을 그대로 평균"하는 것과 "중요도 가중 \(\rho\)"을
붙인 평균을 **동시에** 그려서, `편향은 없애주되 대가로 분산을 극단적으로
키운다`는 중요도 샘플링의 본질을 눈으로 확인합니다.

## 1. 환경: 5칸 회랑 (5.2절과 동일)

칸 0~4, 0과 4가 종료 상태(\(-1\)/\(+1\)), 시작 칸 2,
\(\gamma=0.9\). `step`은 (다음 상태, 보상, 종료 여부)를 반환합니다.

In [1]:
import random
import numpy as np

GAMMA = 0.9

def step(s, a):
    ns = s - 1 if a == 0 else s + 1
    ns = max(0, min(4, ns))
    if ns == 0:
        return ns, -1.0, True
    if ns == 4:
        return ns, 1.0, True
    return ns, 0.0, False

In [2]:
# 행동 정책 b(50-50)로 한 에피소드 샘플링: [(s, a, r), ...]
def sample_episode(rng, start=2, n_max=200):
    s, ep = start, []
    for _ in range(n_max):
        a = 0 if rng.random() < 0.5 else 1
        ns, r, done = step(s, a)
        ep.append((s, a, r))
        s = ns
        if done:
            break
    return ep

# 목표 정책 pi(항상 오른쪽)의 정확한 가치 — 역순 대입(Chapter 4)
V_pi = [0.729, 0.810, 0.900, 1.000, 0.0]
print(f"V^pi(2) = {V_pi[2]:.3f}   (정확값, 역순 대입)")
print("V^b(2)  = 0.0              (중앙 시작 -> 좌/우 대칭으로 상쇄)")

V^pi(2) = 0.900   (정확값, 역순 대입)
V^b(2)  = 0.0              (중앙 시작 -> 좌/우 대칭으로 상쇄)


## 2. 세 가지 추정치 비교 (책의 표 재현)

\(b\)(50-50)의 에피소드만 모아서:

- **그냥 평균** \(\bar{G}_0\) → \(V^b(2)=0\)로 수렴 (\(V^\pi\)의
  **편향된** 추정)
- **중요도 가중 평균** \(\overline{\rho G_0}\) → \(V^\pi(2)=0.9\)로
  수렴 (**무편향**이나 분산 큼)

스텝별 비율은 \(\pi=\)오른쪽, \(b=\)50-50이므로 오른쪽이면 \(2\),
왼쪽이면 \(0\). 즉 **왼쪽이 한 번이라도 나오면 \(\rho=0\)**,
오른쪽만 \(L\)번이면 \(\rho=2^L\).

In [3]:
def estimate(rng, n_episodes, start=2):
    # 세 가지 추정치: (1) 그냥 평균 (2) 중요도 가중 평균
    naive, isw = [], []
    for _ in range(n_episodes):
        ep = sample_episode(rng, start)
        G = 0.0
        for t in reversed(range(len(ep))):
            G = ep[t][2] + GAMMA * G
        naive.append(G)                          # 편향됨: V^b(start)로 수렴
        all_right = all(a == 1 for _, a, _ in ep)
        L = len(ep)
        rho = (2.0 ** L) if all_right else 0.0   # b=50-50, pi=right
        isw.append(rho * G)                      # 편향 없음: V^pi(start)로 수렴
    return np.mean(naive), np.std(naive), np.mean(isw), np.std(isw)

print(f"{'N':>6}  {'naive mean':>10}  {'naive sd':>10}  {'IS mean':>10}  {'IS sd':>10}")
for N in [10, 100, 1000, 10000, 20000]:
    m_n, s_n, m_i, s_i = estimate(random.Random(1), N)
    print(f"{N:>6}  {m_n:>+10.3f}  {s_n:>10.3f}  {m_i:>+10.3f}  {s_i:>10.3f}")
# 기대: naive -> 0.0 (V^b), IS -> 0.9 (V^pi), IS의 sd가 훨씬 큼

     N  naive mean    naive sd     IS mean       IS sd
    10      -0.440       0.703      +0.720       1.440
   100      -0.011       0.776      +0.900       1.559
  1000      +0.033       0.767      +0.878       1.546
 10000      +0.002       0.775      +0.887       1.551
 20000      +0.004       0.774      +0.892       1.554


## 3. 수렴 경로 — 3패널 그림 (본문 ch05_3_importance_sampling.svg)

왼쪽: 그냥 평균이 **\(V^b=0\)**에 수렴 (\(V^\pi=0.9\)의 편향된 추정).
가운데: 중요도 가중치가 붙자 평균은 **\(V^\pi=0.9\)**로 수렴하지만,
개별 \(\rho G_0\) 값(산점)이 0과 3.6 사이에서 극단적으로 퍼져 있음 —
분산 폭증의 직접 증거.
오른쪽: \(\text{Var}[\rho]=2^L-1\)이 에피소드 길이 \(L\)에 대해
**지수적으로** 커지는 모습.

In [4]:
import matplotlib
matplotlib.use("Agg")
from matplotlib import font_manager
import matplotlib.pyplot as plt
# 한국어 라벨을 위한 CJK 폰트 (없으면 DejaVu Sans로 fallback)
kr = [f.name for f in font_manager.fontManager.ttflist if "Noto Sans CJK KR" in f.name]
if kr: plt.rcParams["font.sans-serif"] = [kr[0]]
plt.rcParams["axes.unicode_minus"] = False

# 누적 평균을 에피소드 수 N에 따라
rng = random.Random(1)
n_total = 20000
naive_cum, is_cum, is_vals, naive_vals = [], [], [], []
sn = si = 0.0
for i in range(n_total):
    ep = sample_episode(rng)
    G = 0.0
    for t in reversed(range(len(ep))):
        G = ep[t][2] + GAMMA * G
    all_right = all(a == 1 for _, a, _ in ep)
    rho = (2.0 ** len(ep)) if all_right else 0.0
    iw = rho * G
    sn += G; si += iw
    naive_cum.append(sn / (i + 1))
    is_cum.append(si / (i + 1))
    naive_vals.append(G)
    is_vals.append(iw)
N_arr = np.arange(1, n_total + 1)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# (a) 그냥 평균 -> V^b=0 (편향)
axes[0].plot(N_arr, naive_cum, color="tab:red", lw=1.5, label="Simple average $\\bar{G}$")
axes[0].axhline(0.0, color="gray", ls="--", lw=1, label="$V^b(2)=0$ (convergence point)")
axes[0].axhline(0.9, color="tab:blue", ls=":", lw=1.5, label="$V^\\pi(2)=0.9$ (true value)")
axes[0].set_xlabel("Number of episodes $N$"); axes[0].set_ylabel("Average return")
axes[0].set_ylim(-2, 2)
axes[0].set_title("(a) Simple average: converges to $V^b=0$ (biased)")
axes[0].legend(loc="lower right", fontsize=8)

# (b) 중요도 가중 -> V^pi=0.9 (무편향, 분산 큼)
# 개별 점만 시각화(파일 용량 절감): 균등한 부분집합
step_k = max(1, n_total // 1500)   # 균등 부분집합 (파일 용량 절감)
axes[1].scatter(N_arr[::step_k], is_vals[::step_k], s=3, color="tab:blue", alpha=0.3, label="$\\rho G_0$ (individual, subsample)")
axes[1].plot(N_arr, is_cum, color="tab:red", lw=1.8, label="Importance-weighted average $\\overline{\\rho G}$")
axes[1].axhline(0.9, color="green", ls=":", lw=1.5, label="$V^\\pi(2)=0.9$ (true value)")
axes[1].set_xlabel("Number of episodes $N$"); axes[1].set_ylabel("Value")
axes[1].set_ylim(-1, 4)
axes[1].set_title("(b) Importance weighting: converges to $V^\\pi=0.9$ (unbiased, high variance)")
axes[1].legend(loc="upper right", fontsize=8)

# (c) Var[rho] = 2^L - 1 (지수 폭증)
L_arr = np.arange(1, 13)
var_rho = 2.0 ** L_arr - 1.0
axes[2].semilogy(L_arr, var_rho, "o-", color="tab:orange", label="$\\text{Var}[\\rho]=2^L-1$")
axes[2].set_xlabel("Episode length $L$"); axes[2].set_ylabel("$\\text{Var}[\\rho]$ (log)")
axes[2].set_title("(c) Variance blow-up: $\\text{Var}[\\rho]=2^L-1$ (exponential)")
axes[2].grid(alpha=0.3); axes[2].legend(fontsize=8)

fig.suptitle("Two failure modes of importance sampling: (a) bias  (b) variance blow-up  (c) variance by length", y=1.02)
fig.tight_layout()

IMG = "/home/smhan/book-ml/kor/src/images"
fig.savefig(IMG + "/ch05_3_importance_sampling.svg", bbox_inches="tight")
print(f"저장: {IMG}/ch05_3_importance_sampling.svg")
plt.show()

# 검증: 전체 평균/분산이 이론값에 가까운지
print(f"나머지 확인: naive mean -> {np.mean(naive_vals):.3f} (이론 0.0),  "
      f"IS mean -> {np.mean(is_vals):.3f} (이론 0.9),  IS sd -> {np.std(is_vals):.3f}")

저장: /home/smhan/book-ml/kor/src/images/ch05_3_importance_sampling.svg
나머지 확인: naive mean -> 0.004 (이론 0.0),  IS mean -> 0.892 (이론 0.9),  IS sd -> 1.554


## 4. \(\rho\) 분포가 길이에 따라 퍼지는 모습

실제 5칸 회랑(시작 2)에서는 오른쪽만 \(L=2\)번이면 종료하므로
\(\rho \in \{0, 4\}\)만 나옵니다(\(P(\\rho=4)=0.25\)).
그런데 **에피소드가 길어질수록** \(\text{Var}[\\rho]=2^L-1\)이
지수적으로 커진다는 것이 off-policy의 본질적 난제입니다.

In [5]:
rng = random.Random(2)
rhos = []
for _ in range(200000):
    ep = sample_episode(rng)
    L = len(ep)
    all_right = all(a == 1 for _, a, _ in ep)
    rhos.append(2.0 ** L if all_right else 0.0)
rhos = np.array(rhos)
print(f"P(rho=4) = {(rhos == 4).mean():.3f}   P(rho=0) = {(rhos == 0).mean():.3f}")
print(f"E[rho] = {rhos.mean():.3f}  (이론 1)")
print(f"Var[rho] = {rhos.var():.3f}  (이론 2^2-1 = 3)")
# L이 커질수록 Var[rho] = 2^L - 1 (지수적 폭증)
for L in [1, 2, 3, 4, 6, 8, 10]:
    print(f"L={L:2d}  P(rho>0)={2.0**(-L):.5f}  Var[rho]={2.0**L - 1:.0f}")

P(rho=4) = 0.250   P(rho=0) = 0.750
E[rho] = 1.001  (이론 1)
Var[rho] = 3.002  (이론 2^2-1 = 3)
L= 1  P(rho>0)=0.50000  Var[rho]=1
L= 2  P(rho>0)=0.25000  Var[rho]=3
L= 3  P(rho>0)=0.12500  Var[rho]=7
L= 4  P(rho>0)=0.06250  Var[rho]=15
L= 6  P(rho>0)=0.01562  Var[rho]=63
L= 8  P(rho>0)=0.00391  Var[rho]=255
L=10  P(rho>0)=0.00098  Var[rho]=1023


## 5. continuing task에서 MC가 안 되는 이유

매 스텝 보상 \(+1\), \(\gamma=0.9\), **ending 없음**.
진짜 리턴은 \(1/(1-\\gamma)=10\)인데, \(H\) 스텝으로 절단하면
\(G^{(H)} = \\sum_{k=0}^{H-1}\\gamma^k\)에 그칩니다 — H를 사람이
정해야 하고, H가 짧으면 편향, 길면 분산이 커집니다.

In [6]:
# 매 스텝 보상 +1, gamma=0.9, ending 없음 -> 진짜 리턴 = 1/(1-gamma) = 10
print(f"진짜 리턴 1/(1-0.9) = {1/(1-GAMMA):.4f}")
for H in [5, 20, 50, 200]:
    Gt = sum(GAMMA ** k * 1.0 for k in range(H))
    print(f"H={H:4d}  절단 리턴 G^H = {Gt:.4f}")
# H를 "에피소드 끝"으로 대체할 수 없으므로 사람이 H를 정해야 함
# (Chapter 6 TD 학습이 이 문제를 "한 스텝만 보고 갱신"으로 우회)

진짜 리턴 1/(1-0.9) = 10.0000
H=   5  절단 리턴 G^H = 4.0951
H=  20  절단 리턴 G^H = 8.7842
H=  50  절단 리턴 G^H = 9.9485
H= 200  절단 리턴 G^H = 10.0000
